# Kronos Stock Predictor — 03 预测演示

加载训练好的模型进行 K 线预测并可视化

In [ ]:
import sys; sys.path.insert(0, '..')
import torch, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
from config.default_config import Config
from config.model_configs import build_tokenizer_config, build_model_config
from model.kronos_tokenizer import KronosTokenizer
from model.kronos_model import Kronos
from model.predictor import KronosPredictor
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
c = Config()
tok_cfg = build_tokenizer_config('mini')
tokenizer = KronosTokenizer(**tok_cfg)
tokenizer.load_state_dict(torch.load('./outputs/tokenizer/best_model.pt', map_location='cpu')['model_state_dict'])
tokenizer.eval()

model_cfg = build_model_config('mini')
model = Kronos(**model_cfg)
model.load_state_dict(torch.load('./outputs/predictor/best_model.pt', map_location='cpu')['model_state_dict'])
model.eval()

predictor = KronosPredictor(model, tokenizer, device='cuda:0', max_context=60)
print('Models loaded!')

In [ ]:
# 加载测试数据
with open('./data/processed/test_data.pkl', 'rb') as f:
    test_data = pickle.load(f)
df = test_data[list(test_data.keys())[0]]
print(f'Data: {len(df)} days, {df.index[0].date()} → {df.index[-1].date()}')

In [ ]:
# 预测
lookback = 60; pred_len = 30
x_df = df.iloc[-lookback:][c.feature_list]
x_ts = df.index[-lookback:]
y_ts = pd.date_range(df.index[-1], periods=pred_len+1, freq='B')[1:]

pred_df = predictor.predict(
    df=x_df, x_timestamp=x_ts, y_timestamp=y_ts,
    pred_len=pred_len, T=0.6, top_p=0.9, sample_count=10
)
print('Prediction complete!')

In [ ]:
# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# 价格曲线
hist_close = df['close'].iloc[-lookback:]
hist_dates = df.index[-lookback:]
ax1.plot(hist_dates, hist_close, 'b-', linewidth=1.5, label='Historical')
ax1.plot(pred_df.index, pred_df['close'], 'r--', linewidth=1.5, marker='o', label='Predicted')
ax1.fill_between(pred_df.index, pred_df['low'], pred_df['high'], alpha=0.2, color='red')
ax1.axvline(x=hist_dates[-1], color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Date'); ax1.set_ylabel('Close Price')
ax1.set_title(f'Kronos Stock Prediction ({pred_len} days ahead)')
ax1.legend(); ax1.grid(True, alpha=0.3)

# 收益率分布
returns = pred_df['close'].pct_change().dropna()
ax2.bar(range(len(returns)), returns.values * 100, alpha=0.7, color='steelblue')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_xlabel('Prediction Step'); ax2.set_ylabel('Daily Return (%)')
ax2.set_title('Predicted Daily Returns')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()